# **Baseline Model**

In [2]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

In [8]:
# Loading the dataset
df_train = pd.read_csv("../dataset/UNSW_NB15_training-set.csv")
df_test = pd.read_csv("../dataset/UNSW_NB15_testing-set.csv")
pd.set_option('display.max_columns', None)

In [ ]:
# Separate features and target
target = "label"

drop_cols = ["id", "attack_cat", target]

X_train = df_train.drop(columns=drop_cols)
y_train = df_train[target]

X_test = df_test.drop(columns=drop_cols)
y_test = df_test[target]

In [10]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (175341, 42)
X_test : (82332, 42)
y_train: (175341,)
y_test : (82332,)


In [ ]:
# Identify numerical and categorical features

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

Categorical features:
['proto', 'service', 'state']

Number of numerical features: 39
Number of categorical features: 3


In [ ]:
# Create preprocessing pipeline

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [ ]:
# Creating Logistic Regression pipeline

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
# Training

baseline_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sttl', 'dttl', 'sload',
                                                   'dload', 'sloss', 'dloss',
                                                   'sinpkt', 'dinpkt', 'sjit',
                                                   'djit', 'swin', 'stcpb',
                                                   'dtcpb', 'dwin', 'tcprtt',
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm', ...]),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['proto', 'service',
                                                   'state'])])),
                ('model', LogisticRegression(max_iter=1000, n_jobs=-1))])

In [ ]:
# Making Predictions    

y_pred = baseline_model.predict(X_test)

y_prob = baseline_model.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluating

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

Accuracy : 0.8097
Precision: 0.7535
Recall   : 0.9724
F1 Score : 0.8491
ROC-AUC  : 0.9556
PR-AUC   : 0.9661


In [17]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.61      0.74     37000
           1       0.75      0.97      0.85     45332

    accuracy                           0.81     82332
   macro avg       0.85      0.79      0.80     82332
weighted avg       0.84      0.81      0.80     82332



In [18]:
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)


Confusion Matrix:
[[22579 14421]
 [ 1249 44083]]


### **Insight :**

The Logistic Regression baseline achieves strong attack detection capability, with 97.24% recall and 95.56% ROC-AUC. However, its 61% recall for normal traffic and 14,421 false positives indicate that the model is overly aggressive in classifying traffic as malicious.